In [44]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, MinHashLSH


In [47]:
spark = (
    SparkSession.builder
    .appName("ArxivSimilarAbstracts")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.default.parallelism", "64")
    .config("spark.driver.memory", "6g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")


In [49]:
file_path = '/home/jupyter/arxiv-metadata-oai-snapshot.json'
sample_size = 30000

raw = (
    spark.read.json(file_path)
    .select("id", "title", "authors", "abstract")
    .where(F.col("abstract").isNotNull())
    .limit(sample_size)
)

In [ ]:
cleaned = raw.withColumn("clean_text", F.lower(F.col("abstract")))

cleaned = cleaned.withColumn(
    "clean_text",
    F.regexp_replace("clean_text", r"http\S+|www\.\S+|\S+@\S+", " ")
)
cleaned = cleaned.withColumn(
    "clean_text",
    F.regexp_replace("clean_text", r"\$.*?\$", " ")
)
cleaned = cleaned.withColumn(
    "clean_text",
    F.regexp_replace("clean_text", r"\\[a-zA-Z]+(\{.*?\})?", " ")
)
cleaned = cleaned.withColumn(
    "clean_text",
    F.regexp_replace("clean_text", r"<[^>]+>", " ")
)
cleaned = cleaned.withColumn(
    "clean_text",
    F.regexp_replace("clean_text", r"[^a-z0-9\s]", " ")
)
cleaned = cleaned.withColumn(
    "clean_text",
    F.regexp_replace("clean_text", r"\s+", " ")
)

cleaned = cleaned.select("id", "title", "authors", F.trim("clean_text").alias("clean_text"))

In [51]:
tokenizer = RegexTokenizer(
    inputCol="clean_text",
    outputCol="tokens",
    pattern=r"\s+",
    gaps=True,
    minTokenLength=3
)

tokenized = tokenizer.transform(cleaned)

remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

tokens_df = (
    remover.transform(tokenized)
    .withColumn("filtered_tokens", F.array_distinct("filtered_tokens"))
    .where(F.size("filtered_tokens") > 5)
    .select("id", "title", "authors", "filtered_tokens")
    .repartition(64)
    .cache()
)

print(tokens_df.count())
tokens_df.show(5, truncate=80)

29934
+---------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|       id|                                                                           title|                                                                         authors|                                                                 filtered_tokens|
+---------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|0704.0893|                Topological phase for spin-orbit transformations on a laser beam|                       C.E.R. Souza, J.A.O. Huguenin, P. Milman, and A.Z. Khoury|[investigate, topological, phase, associated, double, co

In [52]:
cv = CountVectorizer(
    inputCol="filtered_tokens",
    outputCol="features",
    binary=True,
    minDF=2.0
)

cv_model = cv.fit(tokens_df)

features_df = (
    cv_model.transform(tokens_df)
    .select("id", "title", "authors", "filtered_tokens", "features")
    .cache()
)

print("Vocabulary size:", len(cv_model.vocabulary))

Vocabulary size: 26581


In [53]:
JACCARD_THRESHOLD = 0.5
DISTANCE_THRESHOLD = 1.0 - JACCARD_THRESHOLD

mh = MinHashLSH(
    inputCol="features",
    outputCol="hashes",
    numHashTables=20,
    seed=42
)

mh_model = mh.fit(features_df)

joined = mh_model.approxSimilarityJoin(
    features_df,
    features_df,
    DISTANCE_THRESHOLD,
    distCol="jaccard_distance"
)

In [54]:
similar_pairs = (
    joined
    .where(F.col("datasetA.id") < F.col("datasetB.id"))
    .select(
        F.col("datasetA.id").alias("id1"),
        F.col("datasetB.id").alias("id2"),
        F.col("datasetA.title").alias("title1"),
        F.col("datasetB.title").alias("title2"),
        (1.0 - F.col("jaccard_distance")).alias("jaccard_similarity")
    )
    .orderBy(F.desc("jaccard_similarity"))
)

similar_pairs.show(20, truncate=100)

+---------+---------+----------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+------------------+
|      id1|      id2|                                                                                              title1|                                                                                              title2|jaccard_similarity|
+---------+---------+----------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+------------------+
|0708.0121|0708.0132|Discussion of ``2004 IMS Medallion Lecture: Local Rademacher\n  complexities and oracle inequalit...|Discussion of ``2004 IMS Medallion Lecture: Local Rademacher\n  complexities and oracle inequalit...|               1.0|
|0708.0089|0708.0121|Discuss

In [56]:
output_path = "results/arxiv_similar_pairs_parquet"

similar_pairs.write.mode("overwrite").parquet(output_path)

In [58]:
similar_pairs_saved = spark.read.parquet("results/arxiv_similar_pairs_parquet")

In [57]:
similar_pairs.coalesce(1).write.mode("overwrite").option("header", True).csv(
    "results/arxiv_similar_pairs_csv"
)

In [65]:
print("Hello, World!")

Hello, World!


In [ ]:
exact_pairs = (
    joined
    .where(F.col("datasetA.id") < F.col("datasetB.id"))
    .withColumn(
        "intersection_size",
        F.size(F.array_intersect("datasetA.filtered_tokens", "datasetB.filtered_tokens"))
    )
    .withColumn(
        "union_size",
        F.size(F.array_union("datasetA.filtered_tokens", "datasetB.filtered_tokens"))
    )
    .withColumn(
        "exact_jaccard",
        F.col("intersection_size") / F.col("union_size")
    )
    .where(F.col("exact_jaccard") >= JACCARD_THRESHOLD)
    .select(
        F.col("datasetA.id").alias("id1"),
        F.col("datasetB.id").alias("id2"),
        F.col("datasetA.title").alias("title1"),
        F.col("datasetB.title").alias("title2"),
        "exact_jaccard"
    )
    .orderBy(F.desc("exact_jaccard"))
)

exact_pairs.show(20, truncate=100)